In [1]:
# Packages
import duckdb
import os
import pandas as pd

# Local functions
from mimic_pipeline.duck import *
from mimic_pipeline.icustays import *
from mimic_pipeline.io_registry import register_parquet

# Constants
CLEAR_DB_DATA = True

## Read Data

In [2]:
# Change these to suit your needs
base_path = "data/"
directory_map = {
    # core
    "ADMISSIONS": "base_parquet/admissions.parquet",
    "PATIENTS": "base_parquet/patients.parquet",

    # hospital
    "DIAGNOSES_ICD": "base_parquet/diagnoses_icd.parquet",
    "D_ICD_DIAGNOSES": "base_parquet/d_icd_diagnoses.parquet",

    # icu
    "ICUSTAYS": "base_parquet/icustays.parquet",
    "INPUTEVENTS": "base_parquet/inputevents.parquet",
    "OUTPUTEVENTS": "base_parquet/outputevents.parquet",
    "CHARTEVENTS": "base_parquet/chartevents.parquet",
    "D_ITEMS": "base_parquet/d_items.parquet",
}
filepath_map = {
    key: base_path + value for key, value in directory_map.items()
}

In [3]:
# Clear previous ddb file if needed
if CLEAR_DB_DATA:
    ddb_path = os.path.join(base_path, "preprocessing.duckdb")
    if os.path.exists(ddb_path):
        os.remove(ddb_path)
    ddb_path = os.path.join(base_path, "preprocessing.duckdb.wal")
    if os.path.exists(ddb_path):
        os.remove(ddb_path)

# Prep ddb connection for data exploration
ddb = configure_duckdb(database_name="data/preprocessing.duckdb", memory_limit="8GB", temp_directory=base_path)
register_parquet(ddb, filepath_map)

Done: Loaded ADMISSIONS table in 0.50s
Done: Loaded PATIENTS table in 0.37s
Done: Loaded DIAGNOSES_ICD table in 0.28s
Done: Loaded D_ICD_DIAGNOSES table in 0.07s
Done: Loaded ICUSTAYS table in 0.07s
Done: Loaded INPUTEVENTS table in 3.40s
Done: Loaded OUTPUTEVENTS table in 1.20s
Done: Loaded CHARTEVENTS table in 14.63s
Done: Loaded D_ITEMS table in 0.02s


In [4]:
# Gather preprocessed data
run(ddb, f"CREATE OR REPLACE TABLE COHORT AS SELECT * FROM 'data/preprocessing_checkpoints/cohort.parquet'", "Loaded COHORT table")
run(ddb, f"CREATE OR REPLACE TABLE CLEAN_CHARTEVENTS AS SELECT * FROM 'data/preprocessing_checkpoints/clean_chartevents.parquet'", "Loaded CLEAN_CHARTEVENTS table")
run(ddb, f"CREATE OR REPLACE TABLE CLEAN_INPUTEVENTS AS SELECT * FROM 'data/preprocessing_checkpoints/clean_inputevents.parquet'", "Loaded CLEAN_INPUTEVENTS table")
run(ddb, f"CREATE OR REPLACE TABLE CLEAN_OUTPUTEVENTS AS SELECT * FROM 'data/preprocessing_checkpoints/clean_outputevents.parquet'", "Loaded CLEAN_OUTPUTEVENTS table")

Done: Loaded COHORT table in 0.58s
Done: Loaded CLEAN_CHARTEVENTS table in 14.19s
Done: Loaded CLEAN_INPUTEVENTS table in 4.17s
Done: Loaded CLEAN_OUTPUTEVENTS table in 1.08s


## Data Reduction

### CHARTEVENTS
#### Cleaning Summary
- Filter to values with 0 or null warning
- Drop storetime since it is not relevant
- Keep only quantitative measurements
    - Qualitative would be interesting later, but is harder to make immediate meaning of with a model without addition of NLP techniques

In [5]:
count(ddb, "CHARTEVENTS")

   └─ Counted rows in CHARTEVENTS: 329,499,788 rows


329499788

In [6]:
count(ddb, "CLEAN_CHARTEVENTS")

   └─ Counted rows in CLEAN_CHARTEVENTS: 322,691,706 rows


322691706

### ICUSTAYS
First, a look at this data:

In [7]:
peek(ddb, "ICUSTAYS", 5)

┌────────────┬──────────┬──────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┐
│ subject_id │ hadm_id  │ stay_id  │   first_careunit    │    last_careunit    │       intime        │       outtime       │         los         │
│   int64    │  int64   │  int64   │       varchar       │       varchar       │      timestamp      │      timestamp      │       double        │
├────────────┼──────────┼──────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┤
│   17867402 │ 24528534 │ 31793211 │ Trauma SICU (TSICU) │ Trauma SICU (TSICU) │ 2154-03-03 04:11:00 │ 2154-03-04 18:16:56 │  1.5874537037037035 │
│   14435996 │ 28960964 │ 31983544 │ Trauma SICU (TSICU) │ Trauma SICU (TSICU) │ 2150-06-19 17:57:00 │ 2150-06-22 18:33:54 │            3.025625 │
│   17609946 │ 27385897 │ 33183475 │ Trauma SICU (TSICU) │ Trauma SICU (TSICU) │ 2138-02-05 18:54:00 │ 2138-02-15 12:4

Because the paper references a total of 71791 distinct ICU admission records with an average stay of 4 days I investigated this table.  
According to the tables we downloaded and verified for ICUSTAYS, there are 76,540 records with an average LOS of 3.47 days.

In [8]:
count(ddb, "ICUSTAYS")

query = f"""
    SELECT AVG(los)
    FROM ICUSTAYS
    """
ddb.query(query).show()

   └─ Counted rows in ICUSTAYS: 76,540 rows
┌───────────────────┐
│     avg(los)      │
│      double       │
├───────────────────┤
│ 3.472210471981478 │
└───────────────────┘



We also confirm there are no duplicates in these figures.

In [9]:
ddb.execute("""
            CREATE OR REPLACE VIEW ICUSTAYS_DEDUP AS
            SELECT DISTINCT stay_id
            FROM ICUSTAYS
            """)
count(ddb, "ICUSTAYS_DEDUP")

   └─ Counted rows in ICUSTAYS_DEDUP: 76,540 rows


76540

It would stand to reason the ICUSTAYS were reduced via the cleaned data.

In [ ]:
peek(ddb, "COHORT", 5)
count(ddb, "COHORT")

┌────────────┬──────────┬─────────────────────┬─────────────────────┬─────────────────────┬──────────────────────┐
│ subject_id │ hadm_id  │      admittime      │      dischtime      │ survival_time_hours │ died_within_icu_stay │
│   int64    │  int64   │      timestamp      │      timestamp      │       double        │        int32         │
├────────────┼──────────┼─────────────────────┼─────────────────────┼─────────────────────┼──────────────────────┤
│   16358985 │ 24980209 │ 2188-12-31 14:43:00 │ 2189-01-01 18:48:00 │                28.0 │                    0 │
│   10234917 │ 26495171 │ 2136-07-02 10:41:00 │ 2136-07-06 16:50:00 │               102.0 │                    0 │
│   12079093 │ 24857875 │ 2138-11-20 21:20:00 │ 2138-11-24 20:47:00 │                95.0 │                    0 │
│   10534316 │ 29068641 │ 2170-08-17 18:55:00 │ 2170-08-29 21:50:00 │               291.0 │                    0 │
│   11900721 │ 20523250 │ 2188-02-28 18:09:00 │ 2188-03-07 15:54:00 │           

In [11]:
peek(ddb, "CLEAN_INPUTEVENTS", 5)
peek(ddb, "CLEAN_OUTPUTEVENTS", 5)
peek(ddb, "CLEAN_CHARTEVENTS", 5)

┌────────────┬──────────┬──────────┬─────────────────────┬─────────────────────┬─────────────────────┬────────┬────────┬───────────┬────────┬─────────┬─────────┬─────────────┬────────────────────────┬────────────────────────────┬───────────────────────────────┬──────────────────────────┬───────────────┬─────────────┬────────────────┬───────────┬────────────────────┬──────────────┬───────────────────┬────────────────┬──────────────┐
│ subject_id │ hadm_id  │ stay_id  │      starttime      │       endtime       │      storetime      │ itemid │ amount │ amountuom │  rate  │ rateuom │ orderid │ linkorderid │   ordercategoryname    │ secondaryordercategoryname │ ordercomponenttypedescription │ ordercategorydescription │ patientweight │ totalamount │ totalamountuom │ isopenbag │ continueinnextdept │ cancelreason │ statusdescription │ originalamount │ originalrate │
│   int64    │  int64   │  int64   │      timestamp      │      timestamp      │      timestamp      │ int64  │ double │  varcha

In [ ]:
filter_icustays_on_cohort(ddb, icu="ICUSTAYS", cohort="COHORT", output_table="ICUSTAYS_COHORT")

peek(ddb, "ICUSTAYS_COHORT")
count(ddb, "ICUSTAYS_COHORT")

┌────────────┬──────────┬──────────┬──────────────────────────────────────────────────┬──────────────────────────────────────────────────┬─────────────────────┬─────────────────────┬────────────────────┐
│ subject_id │ hadm_id  │ stay_id  │                  first_careunit                  │                  last_careunit                   │       intime        │       outtime       │        los         │
│   int64    │  int64   │  int64   │                     varchar                      │                     varchar                      │      timestamp      │      timestamp      │       double       │
├────────────┼──────────┼──────────┼──────────────────────────────────────────────────┼──────────────────────────────────────────────────┼─────────────────────┼─────────────────────┼────────────────────┤
│   13513470 │ 26950517 │ 35685709 │ Trauma SICU (TSICU)                              │ Trauma SICU (TSICU)                              │ 2177-03-14 22:47:00 │ 2177-03-15 16:00:30 │ 0

76513

In [13]:
# Filter ICUSTAYS_COHORT to only those with data in all three event tables
filter_icustays_with_all_events(ddb, "ICUSTAYS_COHORT", "CLEAN_CHARTEVENTS", "CLEAN_OUTPUTEVENTS", "CLEAN_INPUTEVENTS")

count(ddb, "ICUSTAYS_COHORT_REDUCED")

Filtered ICUSTAYS_COHORT to ICUSTAYS_COHORT_REDUCED with data in all three event tables.
   └─ Counted rows in ICUSTAYS_COHORT_REDUCED: 74,062 rows


74062

Using a 24-hour window from the first instance of ICU data is apparently a common restriction.
- Most notably, the [MIMIC Code Repository](https://github.com/MIT-LCP/mimic-code/tree/main/mimic-iv/concepts/firstday) and [MIMIC-Extract Pipeline](https://github.com/MLforHealth/MIMIC_Extract) use this restriction for ICU data
- These apparently follow the lead of scoring methodologies such as [SAPS-II](https://pubmed.ncbi.nlm.nih.gov/8254858/) and [APACHE II (The linked paper uses APACHE II and summarizes its methodology)](https://pmc.ncbi.nlm.nih.gov/articles/PMC10060092/)
- [OASIS also makes use of information within the first 24 hours of ICU admission](https://pubmed.ncbi.nlm.nih.gov/23660729/)

In [14]:
# Restrict to ICU stays with data in all three event tables within first 24 hours
filter_icustays_first_24h(ddb, "ICUSTAYS_COHORT", "CLEAN_CHARTEVENTS", "CLEAN_OUTPUTEVENTS", "CLEAN_INPUTEVENTS")

count(ddb, "ICUSTAYS_COHORT_REDUCED_24H")

   └─ Counted rows in ICUSTAYS_COHORT_REDUCED_24H: 73,411 rows


73411

Limiting to the first stay is also important from those same studies (may want to cite this).

In [ ]:
filter_icustays_first_stay(ddb, from_table="ICUSTAYS_COHORT_REDUCED_24H", output_table="ICUSTAYS_COHORT_FIRST")

count(ddb, "ICUSTAYS_COHORT_FIRST")

   └─ Counted rows in ICUSTAYS_COHORT_FIRST: 52,021 rows


52021

Reminder of the ICUSTAYS structure

In [18]:
peek(ddb, "ICUSTAYS_COHORT_FIRST", 1)

┌────────────┬──────────┬──────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬───────────────────┬───────┐
│ subject_id │ hadm_id  │ stay_id  │   first_careunit    │    last_careunit    │       intime        │       outtime       │        los        │  rn   │
│   int64    │  int64   │  int64   │       varchar       │       varchar       │      timestamp      │      timestamp      │      double       │ int64 │
├────────────┼──────────┼──────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼───────────────────┼───────┤
│   10008454 │ 20291550 │ 31959184 │ Trauma SICU (TSICU) │ Trauma SICU (TSICU) │ 2110-11-30 17:11:36 │ 2110-12-05 16:48:24 │ 4.983888888888889 │     1 │
└────────────┴──────────┴──────────┴─────────────────────┴─────────────────────┴─────────────────────┴─────────────────────┴───────────────────┴───────┘



This reduces the ICU admission records to far below what was referenced in the study so it is unlikely this reduction was performed. In fact, this is hardly more records than there are number of patients present in the raw `ICUSTAYS` data:

In [19]:
ddb.execute("""
            CREATE OR REPLACE VIEW DISTINCT_ICU_PTS AS
            SELECT DISTINCT subject_id
            FROM ICUSTAYS
            """)
count(ddb, "DISTINCT_ICU_PTS")

   └─ Counted rows in DISTINCT_ICU_PTS: 53,150 rows


53150

It is also somewhat common to restrict to adults (MIT, OASIS, other citations). Testing confirmed this was indeed common because the dataset is already resticted to adults.

In [39]:
# Basing this off of the 24H reduction rather than first appearance in ICU since that removed too many.

# peek(ddb, "ICUSTAYS_COHORT_REDUCED_24H", 1)
# peek(ddb, "ADMISSIONS", 1)
# peek(ddb, "PATIENTS", 1)

query = f"""
    CREATE OR REPLACE TEMP TABLE AGE_THROWAWAY AS
    SELECT
        i.*,
        a.admittime,
        a.dischtime,
        /* MIMIC-IV age recipe: anchor_age + (admit_year - anchor_year) */
        (EXTRACT(YEAR FROM a.admittime) - p.anchor_year + p.anchor_age) AS admission_age
    FROM ICUSTAYS i
    JOIN ADMISSIONS a ON i.hadm_id = a.hadm_id
    JOIN PATIENTS p ON i.subject_id = p.subject_id
    WHERE admission_age >= 18;
    """
ddb.execute(query)

# Without the age filter it is still 73411 in length

peek(ddb, "AGE_THROWAWAY", 1)
count(ddb, "AGE_THROWAWAY")

┌────────────┬──────────┬──────────┬──────────────────────────────────────────────────┬──────────────────────────────────────────────────┬─────────────────────┬─────────────────────┬───────────────────┬─────────────────────┬─────────────────────┬───────────────┐
│ subject_id │ hadm_id  │ stay_id  │                  first_careunit                  │                  last_careunit                   │       intime        │       outtime       │        los        │      admittime      │      dischtime      │ admission_age │
│   int64    │  int64   │  int64   │                     varchar                      │                     varchar                      │      timestamp      │      timestamp      │      double       │      timestamp      │      timestamp      │     int64     │
├────────────┼──────────┼──────────┼──────────────────────────────────────────────────┼──────────────────────────────────────────────────┼─────────────────────┼─────────────────────┼───────────────────┼─────────

76540

In [ ]:
ddb.query("SELECT * FROM D_ITEMS").show()

┌────────┬──────────────────────────────────────┬───────────────────────────────┬────────────────┬─────────────────────┬──────────┬───────────────┬────────────────┬─────────────────┐
│ itemid │                label                 │         abbreviation          │    linksto     │      category       │ unitname │  param_type   │ lownormalvalue │ highnormalvalue │
│ int64  │               varchar                │            varchar            │    varchar     │       varchar       │ varchar  │    varchar    │     int64      │     double      │
├────────┼──────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────────────────┼──────────┼───────────────┼────────────────┼─────────────────┤
│ 220003 │ ICU Admission date                   │ ICU Admission date            │ datetimeevents │ ADT                 │ NULL     │ Date and time │           NULL │            NULL │
│ 220045 │ Heart Rate                           │ HR                            │ cha

In [ ]:
query = f"""
        CREATE OR REPLACE TABLE CE_JOINED AS
        SELECT *
        FROM CHARTEVENTS
        JOIN D_ITEMS
        ON CHARTEVENTS.itemid = D_ITEMS.itemid
        """
ddb.execute(query)

In [ ]:
# Consider what we see the most
query = f"""
    SELECT ce.itemid, ce.label, ce.category, ce.unitname, COUNT(*) AS n
    FROM CE_JOINED ce
    GROUP BY ce.itemid, ce.label, ce.category, ce.unitname
    ORDER BY n DESC
    LIMIT 200;
    """
ddb.query(query).show()

┌────────┬──────────────────────────────────────────┬───────────────────────────┬──────────┬─────────┐
│ itemid │                  label                   │         category          │ unitname │    n    │
│ int64  │                 varchar                  │          varchar          │ varchar  │  int64  │
├────────┼──────────────────────────────────────────┼───────────────────────────┼──────────┼─────────┤
│ 227969 │ Safety Measures                          │ Restraint/Support Systems │ NULL     │ 9184087 │
│ 220045 │ Heart Rate                               │ Routine Vital Signs       │ bpm      │ 6798187 │
│ 220210 │ Respiratory Rate                         │ Respiratory               │ insp/min │ 6728530 │
│ 220277 │ O2 saturation pulseoxymetry              │ Respiratory               │ %        │ 6656949 │
│ 220048 │ Heart Rhythm                             │ Routine Vital Signs       │ NULL     │ 6220746 │
│ 224650 │ Ectopy Type 1                            │ Routine Vital Signs

### INPUTEVENTS

In [ ]:
MINI_INPUTEVENTS.head()

NameError: name 'MINI_INPUTEVENTS' is not defined

In [ ]:
MINI_ADMISSIONS.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,language,marital_status,ethnicity,edregtime,edouttime,hospital_expire_flag
0,14031588,27811953,2126-10-22 20:05:00,2126-10-24 16:25:00,NaT,EW EMER.,EMERGENCY ROOM,HOME,Other,ENGLISH,SINGLE,WHITE,2126-10-22 18:16:00,2126-10-22 22:40:00,0
1,14382425,22526290,2159-03-25 21:51:00,2159-03-28 18:00:00,NaT,EW EMER.,EMERGENCY ROOM,HOME HEALTH CARE,Other,ENGLISH,MARRIED,BLACK/AFRICAN AMERICAN,2159-03-25 20:07:00,2159-03-25 23:05:00,0
2,11005665,25778620,2177-04-15 14:14:00,2177-04-20 14:27:00,NaT,EW EMER.,EMERGENCY ROOM,HOME,Medicare,ENGLISH,DIVORCED,BLACK/AFRICAN AMERICAN,2177-04-15 10:41:00,2177-04-15 15:20:00,0
3,13689211,28948245,2156-05-20 13:31:00,2156-05-21 13:54:00,NaT,AMBULATORY OBSERVATION,PACU,None,Medicare,ENGLISH,WIDOWED,WHITE,NaT,NaT,0
4,11550263,21723333,2189-05-10 18:52:00,2189-05-13 14:00:00,NaT,OBSERVATION ADMIT,EMERGENCY ROOM,HOME,Other,ENGLISH,MARRIED,BLACK/AFRICAN AMERICAN,2189-05-10 07:47:00,2189-05-10 20:33:00,0


In [ ]:
MINI_PATIENTS.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,11759524,F,23,2170,2017 - 2019,NaT
1,19969236,F,61,2167,2011 - 2013,NaT
2,15795343,M,80,2112,2017 - 2019,2112-03-13
3,15417324,F,0,2112,2011 - 2013,NaT
4,10668724,F,39,2131,2008 - 2010,NaT


### OUTPUTEVENTS

In [ ]:
MINI_OUTPUTEVENTS.head()

,subject_id,hadm_id,stay_id,charttime,storetime,itemid,value,valueuom
0,11461300,26462757,31497239,2170-05-13 16:00:00,2170-05-13 17:20:00,226559,250.0,ml
1,15505002,22099788,30471766,2145-11-27 16:00:00,2145-11-27 17:26:00,226560,600.0,ml
2,10115397,28605901,31831959,2133-01-19 21:00:00,2133-01-19 21:41:00,226559,150.0,ml
3,13194166,22036580,39270495,2198-08-09 23:00:00,2198-08-09 23:28:00,226559,30.0,ml
4,13441952,28928219,35630151,2175-11-22 15:00:00,2175-11-22 16:26:00,226606,0.0,ml


In [ ]:
# hourly urine example

# SELECT
#     stay_id,
#     DATE_TRUNC('hour', charttime) AS hour,
#     SUM(value) AS urine_ml
# FROM outputevents
# WHERE itemid IN (
#     226559, 226560, 227488  -- common urine output ITEMIDs
# )
# GROUP BY stay_id, hour
# ORDER BY stay_id, hour;
